# 09 — Extract Hidden ProcedureVRL Video Embeddings for Breakfast

This notebook is the next step after the coarse ProcedureVRL baseline.

Notebook 07 extracted final ProcedureVRL output-level features:

```text
per video: [9871, 16]
```

Those are coarse clip-level class/output features. This notebook tries to extract **hidden video embeddings before the final classification head**.

The goal is:

```text
raw Breakfast videos
→ ProcedureVRL video encoder
→ hidden video embeddings before final head
→ MS-TCN-compatible dataset
```

## 1. Mount Drive and imports

In [37]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

from pathlib import Path
import os
import sys
import json
import time
import shutil
import subprocess

import numpy as np
import pandas as pd

print("Python:", sys.version)
print("cwd:", Path.cwd())

Mounted at /content/drive
Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
cwd: /content/ProcedureVRL


## 2. Configuration

In [38]:
DRIVE_ROOT = Path("/content/drive/MyDrive/mmf_tas_lab_data")

# Previous successful ProcedureVRL full extraction run.
PREV_PROC_RUN_ROOT = (
    DRIVE_ROOT
    / "text_assisted_tas"
    / "breakfast"
    / "procedurevrl"
    / "runs"
    / "procedurevrl_breakfast_full_split1_split1_views16"
)

PREV_MSTCN_ROOT = PREV_PROC_RUN_ROOT / "mstcn_format"
PREV_MANIFEST = PREV_PROC_RUN_ROOT / "used_video_manifest.csv"
PREV_DURATION_META = PREV_PROC_RUN_ROOT / "video_duration_metadata.csv"

BREAKFAST_ROOT = DRIVE_ROOT / "zenodo_ms_tcn_data" / "breakfast"
ORIG_GT_DIR = BREAKFAST_ROOT / "groundTruth"
ORIG_SPLIT_DIR = BREAKFAST_ROOT / "splits"
ORIG_MAPPING_PATH = BREAKFAST_ROOT / "mapping.txt"

RAW_VIDEO_ROOT = DRIVE_ROOT / "breakfast_raw_videos"
RAW_VIDEO_BASE = RAW_VIDEO_ROOT / "videos"

PROCEDUREVRL_REPO = Path("/content/ProcedureVRL")
PROCEDUREVRL_GIT = "https://github.com/facebookresearch/ProcedureVRL.git"
PROCEDUREVRL_CKPT_PATH = DRIVE_ROOT / "procedurevrl" / "checkpoints" / "checkpoint_epoch_00025.pyth"

OUT_ROOT = DRIVE_ROOT / "text_assisted_tas" / "breakfast" / "procedurevrl_hidden"
RUNS_ROOT = OUT_ROOT / "runs"
for p in [OUT_ROOT, RUNS_ROOT]:
    p.mkdir(parents=True, exist_ok=True)

SPLIT_ID = 1

# Start with smoke. After it works, change to "full_split1".
# RUN_MODE = "smoke"
RUN_MODE = "full_split1"

CONFIGS = {
    "smoke": {
        "max_videos": 8,
        "num_ensemble_views": 4,
        "batch_size": 4,
        "num_workers": 0,
    },
    "full_split1": {
        "max_videos": None,
        "num_ensemble_views": 16,
        "batch_size": 4,
        "num_workers": 2,
    },
}

cfg_run = CONFIGS[RUN_MODE]

RUN_NAME = f"procedurevrl_hidden_{RUN_MODE}_split{SPLIT_ID}_views{cfg_run['num_ensemble_views']}"
RUN_ROOT = RUNS_ROOT / RUN_NAME
CSV_DIR = RUN_ROOT / "data_csv"
RAW_OUT_DIR = RUN_ROOT / "raw_outputs"
MSTCN_OUT_ROOT = RUN_ROOT / "mstcn_format"
MSTCN_FEATURE_DIR = MSTCN_OUT_ROOT / "features"
MSTCN_GT_DIR = MSTCN_OUT_ROOT / "groundTruth"
MSTCN_SPLIT_DIR = MSTCN_OUT_ROOT / "splits"

for p in [RUN_ROOT, CSV_DIR, RAW_OUT_DIR, MSTCN_OUT_ROOT, MSTCN_FEATURE_DIR, MSTCN_GT_DIR, MSTCN_SPLIT_DIR]:
    p.mkdir(parents=True, exist_ok=True)

HIDDEN_NPY = RAW_OUT_DIR / "procedurevrl_hidden_video_embeddings.npy"
RUN_LOG_PATH = RUN_ROOT / "hidden_extract_log.txt"

print("RUN_MODE:", RUN_MODE)
print("RUN_ROOT:", RUN_ROOT)
print("HIDDEN_NPY:", HIDDEN_NPY)

required_paths = [
    PREV_PROC_RUN_ROOT,
    PREV_MSTCN_ROOT,
    PREV_MANIFEST,
    PREV_DURATION_META,
    RAW_VIDEO_BASE,
    PROCEDUREVRL_CKPT_PATH,
    ORIG_GT_DIR,
    ORIG_SPLIT_DIR,
    ORIG_MAPPING_PATH,
]

for p in required_paths:
    print(p, "->", p.exists())
    assert p.exists(), f"Missing path: {p}"

print("checkpoint size MB:", PROCEDUREVRL_CKPT_PATH.stat().st_size / 1024**2)

RUN_MODE: full_split1
RUN_ROOT: /content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/breakfast/procedurevrl_hidden/runs/procedurevrl_hidden_full_split1_split1_views16
HIDDEN_NPY: /content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/breakfast/procedurevrl_hidden/runs/procedurevrl_hidden_full_split1_split1_views16/raw_outputs/procedurevrl_hidden_video_embeddings.npy
/content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/breakfast/procedurevrl/runs/procedurevrl_breakfast_full_split1_split1_views16 -> True
/content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/breakfast/procedurevrl/runs/procedurevrl_breakfast_full_split1_split1_views16/mstcn_format -> True
/content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/breakfast/procedurevrl/runs/procedurevrl_breakfast_full_split1_split1_views16/used_video_manifest.csv -> True
/content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/breakfast/procedurevrl/runs/procedurevrl_breakfast_full_split1_split1_views16/video_duration_met

## 3. GPU check

In [39]:
import torch

print("CUDA:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

if RUN_MODE == "full_split1" and not torch.cuda.is_available():
    raise RuntimeError("Full hidden embedding extraction should be run on GPU.")

CUDA: True
GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition


## 4. Clone/reuse ProcedureVRL, install dependencies, and apply compatibility patches

In [40]:
if PROCEDUREVRL_REPO.exists():
    print("ProcedureVRL already exists:", PROCEDUREVRL_REPO)
else:
    !git clone {PROCEDUREVRL_GIT} {PROCEDUREVRL_REPO}

%cd /content/ProcedureVRL

!pip install -q yacs simplejson fvcore iopath decord av einops timm pandas scikit-learn opencv-python ffmpeg-python pytorchvideo ipdb ftfy regex tqdm tabulate
!pip uninstall -y clip -q
!pip install -q git+https://github.com/openai/CLIP.git
!pip install -e . -q

PROCEDUREVRL_LIB = PROCEDUREVRL_REPO / "lib"

for p in [PROCEDUREVRL_REPO, PROCEDUREVRL_LIB]:
    s = str(p)
    if s not in sys.path:
        sys.path.insert(0, s)

os.environ["PYTHONPATH"] = str(PROCEDUREVRL_REPO) + ":" + str(PROCEDUREVRL_LIB) + ":" + os.environ.get("PYTHONPATH", "")

# Required by ProcedureVRL weight conversion.
(PROCEDUREVRL_REPO / "exps").mkdir(parents=True, exist_ok=True)

# Patch np.int in feat_extract.py.
feat_extract_path = PROCEDUREVRL_REPO / "tools" / "feat_extract.py"
if feat_extract_path.exists():
    text = feat_extract_path.read_text()
    patched = text.replace("np.int", "int")
    if patched != text:
        feat_extract_path.write_text(patched)
        print("Patched np.int -> int in feat_extract.py")

# Patch .avi support in HowTo100M loader.
howto_path = PROCEDUREVRL_REPO / "lib" / "datasets" / "howto100m.py"
text = howto_path.read_text()
old = 'for extension in [".webm", ".mkv", ".mp4", ".m4a"]:'
new = 'for extension in [".avi", ".AVI", ".webm", ".mkv", ".mp4", ".m4a"]:'
if old in text:
    howto_path.write_text(text.replace(old, new))
    print("Patched HowTo100M loader for .avi/.AVI")
else:
    print("AVI patch already applied or original pattern not found.")

# Patch ViT alias mismatch.
video_builder_path = PROCEDUREVRL_REPO / "lib" / "models" / "video_model_builder.py"
vit_path = PROCEDUREVRL_REPO / "lib" / "models" / "vit.py"

if video_builder_path.exists():
    text = video_builder_path.read_text()
    old = "from lib.models.vit import vit_base_patch16_224"
    new = "from lib.models.vit import vit_base_patch16_224_develop as vit_base_patch16_224"
    if old in text and new not in text:
        video_builder_path.write_text(text.replace(old, new))
        print("Patched video_model_builder.py vit import")

if vit_path.exists():
    text = vit_path.read_text()
    alias_code = """

# Colab / repo compatibility alias.
try:
    vit_base_patch16_224
except NameError:
    vit_base_patch16_224 = vit_base_patch16_224_develop
"""
    if "vit_base_patch16_224 = vit_base_patch16_224_develop" not in text:
        vit_path.write_text(text + alias_code)
        print("Added vit alias")

# Quick imports.
import clip
import pytorchvideo
import ipdb
print("clip:", clip.__file__)
print("pytorchvideo OK")
print("ipdb OK")

import importlib.util
for mod in ["lib", "lib.datasets", "lib.models", "lib.utils.parser", "decord", "torch"]:
    spec = importlib.util.find_spec(mod)
    print(mod, "->", spec is not None)

ProcedureVRL already exists: /content/ProcedureVRL
/content/ProcedureVRL
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  error: subprocess-exited-with-error
  
  × python setup.py egg_info did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Preparing metadata (setup.py) ... error
error: metadata-generation-failed

× Encountered error while generating package metadata.
╰─> See above for output.

note: This is an issue with the package mentioned above, not pip.
hint: See above for details.
AVI patch already applied or original pattern not found.
clip: /usr/local/lib/python3.12/dist-packages/clip/__init__.py
pytorchvideo OK
ipdb OK
lib -> True
lib.datasets -> True
lib.models -> True
lib.utils.parser -> True
decord -> True
torch -> True


## 5. Build hidden-extraction CSV and manifest

In [41]:
df_manifest_full = pd.read_csv(PREV_MANIFEST)
df_duration = pd.read_csv(PREV_DURATION_META)

print("previous manifest:", df_manifest_full.shape)
print("duration metadata:", df_duration.shape)
display(df_manifest_full.head())
display(df_duration.head())

# Merge duration info into manifest.
duration_cols = ["video_id", "duration_sec", "num_raw_frames", "fps"]
df = df_manifest_full.merge(df_duration[duration_cols], on="video_id", how="left")

missing_duration = df["duration_sec"].isna().sum()
print("missing duration rows:", missing_duration)
assert missing_duration == 0

# For smoke, use the first few videos. For full, use all.
if cfg_run["max_videos"] is not None:
    df_run = df.head(cfg_run["max_videos"]).copy()
else:
    df_run = df.copy()

df_run = df_run.reset_index(drop=True)
df_run["run_index"] = np.arange(len(df_run))

# Build ProcedureVRL test.csv.
csv_rows = []
rel_rows = []

for _, row in df_run.iterrows():
    video_path = Path(row["video_path"])
    assert video_path.exists(), video_path

    rel = video_path.relative_to(RAW_VIDEO_BASE)
    rel_no_ext = str(rel.with_suffix(""))
    duration_sec = float(row["duration_sec"])

    csv_rows.append(f"{rel_no_ext} 0 {duration_sec:.3f}")

    rel_rows.append({
        "video_id": row["video_id"],
        "split": row["split"],
        "video_path": str(video_path),
        "rel_no_ext": rel_no_ext,
        "duration_sec": duration_sec,
        "run_index": int(row["run_index"]),
    })

(CSV_DIR / "test.csv").write_text("\n".join(csv_rows) + "\n")
(CSV_DIR / "train.csv").write_text("\n".join(csv_rows[:1]) + "\n")
(CSV_DIR / "val.csv").write_text("\n".join(csv_rows[:1]) + "\n")

df_run_manifest = pd.DataFrame(rel_rows)
run_manifest_path = RUN_ROOT / "hidden_used_video_manifest.csv"
df_run_manifest.to_csv(run_manifest_path, index=False)

print("videos used:", len(df_run_manifest))
print("CSV:", CSV_DIR / "test.csv")
print("manifest:", run_manifest_path)
print((CSV_DIR / "test.csv").read_text().splitlines()[:5])
display(df_run_manifest.head())

previous manifest: (1712, 10)
duration metadata: (1712, 7)


,split,video_id,person,view,raw_stem,matched,num_matches,video_path,gt_path,i3d_feature_path
0,train,P16_cam01_P16_cereals,P16,cam01,P16_cereals,True,3,/content/drive/MyDrive/mmf_tas_lab_data/breakf...,/content/drive/MyDrive/mmf_tas_lab_data/zenodo...,/content/drive/MyDrive/mmf_tas_lab_data/zenodo...
1,train,P16_cam01_P16_friedegg,P16,cam01,P16_friedegg,True,3,/content/drive/MyDrive/mmf_tas_lab_data/breakf...,/content/drive/MyDrive/mmf_tas_lab_data/zenodo...,/content/drive/MyDrive/mmf_tas_lab_data/zenodo...
2,train,P16_cam01_P16_juice,P16,cam01,P16_juice,True,3,/content/drive/MyDrive/mmf_tas_lab_data/breakf...,/content/drive/MyDrive/mmf_tas_lab_data/zenodo...,/content/drive/MyDrive/mmf_tas_lab_data/zenodo...
3,train,P16_cam01_P16_milk,P16,cam01,P16_milk,True,3,/content/drive/MyDrive/mmf_tas_lab_data/breakf...,/content/drive/MyDrive/mmf_tas_lab_data/zenodo...,/content/drive/MyDrive/mmf_tas_lab_data/zenodo...
4,train,P16_cam01_P16_pancake,P16,cam01,P16_pancake,True,3,/content/drive/MyDrive/mmf_tas_lab_data/breakf...,/content/drive/MyDrive/mmf_tas_lab_data/zenodo...,/content/drive/MyDrive/mmf_tas_lab_data/zenodo...


,video_id,video_path,rel_no_ext,duration_sec,num_raw_frames,fps,split
0,P16_cam01_P16_cereals,/content/drive/MyDrive/mmf_tas_lab_data/breakf...,P16/cam01/P16_cereals,36.600000,549,15.0,train
1,P16_cam01_P16_friedegg,/content/drive/MyDrive/mmf_tas_lab_data/breakf...,P16/cam01/P16_friedegg,462.400000,6936,15.0,train
2,P16_cam01_P16_juice,/content/drive/MyDrive/mmf_tas_lab_data/breakf...,P16/cam01/P16_juice,149.466667,2242,15.0,train
3,P16_cam01_P16_milk,/content/drive/MyDrive/mmf_tas_lab_data/breakf...,P16/cam01/P16_milk,59.200000,888,15.0,train
4,P16_cam01_P16_pancake,/content/drive/MyDrive/mmf_tas_lab_data/breakf...,P16/cam01/P16_pancake,536.266667,8044,15.0,train


missing duration rows: 0
videos used: 1712
CSV: /content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/breakfast/procedurevrl_hidden/runs/procedurevrl_hidden_full_split1_split1_views16/data_csv/test.csv
manifest: /content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/breakfast/procedurevrl_hidden/runs/procedurevrl_hidden_full_split1_split1_views16/hidden_used_video_manifest.csv
['P16/cam01/P16_cereals 0 36.600', 'P16/cam01/P16_friedegg 0 462.400', 'P16/cam01/P16_juice 0 149.467', 'P16/cam01/P16_milk 0 59.200', 'P16/cam01/P16_pancake 0 536.267']


,video_id,split,video_path,rel_no_ext,duration_sec,run_index
0,P16_cam01_P16_cereals,train,/content/drive/MyDrive/mmf_tas_lab_data/breakf...,P16/cam01/P16_cereals,36.600000,0
1,P16_cam01_P16_friedegg,train,/content/drive/MyDrive/mmf_tas_lab_data/breakf...,P16/cam01/P16_friedegg,462.400000,1
2,P16_cam01_P16_juice,train,/content/drive/MyDrive/mmf_tas_lab_data/breakf...,P16/cam01/P16_juice,149.466667,2
3,P16_cam01_P16_milk,train,/content/drive/MyDrive/mmf_tas_lab_data/breakf...,P16/cam01/P16_milk,59.200000,3
4,P16_cam01_P16_pancake,train,/content/drive/MyDrive/mmf_tas_lab_data/breakf...,P16/cam01/P16_pancake,536.266667,4


## 6. Write custom hidden-embedding extraction script

In [42]:
hidden_script_path = PROCEDUREVRL_REPO / "tools" / "feat_extract_hidden.py"

script = r'''
#!/usr/bin/env python3
import json
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from tqdm import tqdm

from lib.datasets import loader
from lib.models import build_model
import lib.utils.checkpoint as cu
import lib.utils.logging as logging
from lib.utils.misc import launch_job
from lib.utils.parser import parse_args, load_config

logger = logging.get_logger(__name__)

_CAPTURE = {}


def _recursive_to_cuda(x):
    if isinstance(x, torch.Tensor):
        return x.cuda(non_blocking=True) if torch.cuda.is_available() else x
    if isinstance(x, list):
        return [_recursive_to_cuda(v) for v in x]
    if isinstance(x, tuple):
        return tuple(_recursive_to_cuda(v) for v in x)
    if isinstance(x, dict):
        return {k: _recursive_to_cuda(v) for k, v in x.items()}
    return x


def _as_numpy_index(video_idx):
    if isinstance(video_idx, torch.Tensor):
        return video_idx.detach().cpu().numpy().astype(np.int64)
    return np.asarray(video_idx, dtype=np.int64)


def _choose_video_embedding_head(model):
    """
    In this ProcedureVRL checkpoint, the useful video embedding projection is:

        model.head = Linear(in_features=768, out_features=512)

    It is NOT a 9871-way classifier. The 9871-dimensional output is produced later
    by comparing the 512-dim video embedding with text/label embeddings.
    """

    all_linear = []

    for name, module in model.named_modules():
        if isinstance(module, nn.Linear):
            all_linear.append({
                "name": name,
                "type": type(module).__name__,
                "in_features": int(module.in_features),
                "out_features": int(module.out_features),
                "repr": repr(module)[:300],
            })

    # Preferred exact target.
    modules = dict(model.named_modules())
    if "model.head" in modules:
        m = modules["model.head"]
        if isinstance(m, nn.Linear):
            target = {
                "name": "model.head",
                "type": type(m).__name__,
                "in_features": int(m.in_features),
                "out_features": int(m.out_features),
                "repr": repr(m)[:300],
            }
            return target, all_linear

    # Fallback: choose a non-text, non-order-transformer 512-dim projection.
    candidates = [
        c for c in all_linear
        if c["out_features"] == 512
        and "text_model" not in c["name"]
        and "order_tfm" not in c["name"]
    ]

    def score(c):
        name = c["name"].lower()
        s = 0
        if "head" in name:
            s -= 100
        if "projection" in name or "proj" in name:
            s -= 20
        if "text" in name:
            s += 100
        if "order" in name:
            s += 100
        return (s, len(name))

    candidates = sorted(candidates, key=score)

    if len(candidates) > 0:
        return candidates[0], all_linear

    raise RuntimeError(
        "Could not find a suitable video embedding head. "
        "Linear layers found: " + json.dumps(all_linear[:120], indent=2)
    )


def _register_embedding_hook(model, target_name):
    """
    We want the OUTPUT of model.head:

        [B, 768] -> model.head -> [B, 512]

    Therefore this is a forward hook, not a pre-forward hook.
    """

    modules = dict(model.named_modules())
    target_module = modules[target_name]

    def forward_hook(module, inputs, output):
        x = output

        if isinstance(x, torch.Tensor):
            _CAPTURE["hidden"] = x.detach()
            _CAPTURE["raw_shape"] = list(x.shape)
            _CAPTURE["target_name"] = target_name
        elif isinstance(x, (tuple, list)) and len(x) > 0 and isinstance(x[0], torch.Tensor):
            _CAPTURE["hidden"] = x[0].detach()
            _CAPTURE["raw_shape"] = list(x[0].shape)
            _CAPTURE["target_name"] = target_name

    handle = target_module.register_forward_hook(forward_hook)
    return handle


def _normalize_hidden_tensor(x):
    if not isinstance(x, torch.Tensor):
        raise TypeError(f"Captured hidden value is not a tensor: {type(x)}")

    # Expected: [B, 512].
    if x.ndim == 2:
        y = x
    elif x.ndim == 3:
        # If token dimension exists, average over tokens.
        y = x.mean(dim=1)
    else:
        y = x.reshape(x.shape[0], -1)

    return y.detach().float().cpu().numpy()


@torch.no_grad()
def perform_hidden_extraction(test_loader, model, cfg):
    model.eval()

    n_clip = int(cfg.TEST.NUM_ENSEMBLE_VIEWS * cfg.TEST.NUM_SPATIAL_CROPS)
    n_samples = len(test_loader.dataset)

    assert n_samples % n_clip == 0, (n_samples, n_clip)

    n_video = n_samples // n_clip

    print(f"n_samples: {n_samples}; n_video: {n_video}; n_clip: {n_clip}")

    target, all_linear = _choose_video_embedding_head(model)

    print("Selected hidden hook target:")
    print(json.dumps(target, indent=2))

    print("First 40 Linear layers:")
    print(json.dumps(all_linear[:40], indent=2))

    handle = _register_embedding_hook(model, target["name"])

    hidden_store = None
    filled = np.zeros((n_video, n_clip), dtype=np.int32)
    raw_hidden_shapes = []

    for cur_iter, (inputs, labels, video_idx, meta) in enumerate(tqdm(test_loader)):
        inputs = _recursive_to_cuda(inputs)
        idx = _as_numpy_index(video_idx)

        _CAPTURE.clear()

        _ = model(inputs)

        if "hidden" not in _CAPTURE:
            raise RuntimeError(
                f"No hidden tensor was captured at iter {cur_iter}. "
                f"Target hook: {target['name']}"
            )

        hidden = _normalize_hidden_tensor(_CAPTURE["hidden"])  # [B, D]
        raw_hidden_shapes.append(_CAPTURE.get("raw_shape"))

        if hidden_store is None:
            hidden_dim = int(hidden.shape[1])
            hidden_store = np.zeros((n_video, n_clip, hidden_dim), dtype=np.float32)
            print("hidden_store shape:", hidden_store.shape)

        if hidden.shape[0] != len(idx):
            raise RuntimeError(f"Batch mismatch: hidden {hidden.shape}, video_idx {idx.shape}")

        video_indices = idx // n_clip
        clip_indices = idx % n_clip

        for b in range(hidden.shape[0]):
            vi = int(video_indices[b])
            ci = int(clip_indices[b])

            if vi < 0 or vi >= n_video:
                raise RuntimeError(f"Bad video index: vi={vi}, n_video={n_video}, raw idx={idx[b]}")

            hidden_store[vi, ci, :] = hidden[b]
            filled[vi, ci] += 1

    handle.remove()

    missing = int((filled == 0).sum())
    duplicates = int((filled > 1).sum())

    print("filled missing:", missing)
    print("filled duplicates:", duplicates)

    if missing != 0:
        raise RuntimeError(f"Some video/clip positions were not filled: {missing}")

    summary = {
        "hook_target": target,
        "n_video": int(n_video),
        "n_clip": int(n_clip),
        "hidden_dim": int(hidden_store.shape[2]),
        "raw_hidden_shapes_seen": raw_hidden_shapes[:20],
        "filled_missing": missing,
        "filled_duplicates": duplicates,
    }

    return hidden_store, summary


def test(cfg):
    logging.setup_logging(cfg.OUTPUT_DIR)

    print("Building model...")
    model = build_model(cfg)

    if torch.cuda.is_available():
        model = model.cuda()

    print("Loading checkpoint...")
    cu.load_test_checkpoint(cfg, model)

    print("Constructing test loader...")
    test_loader = loader.construct_loader(cfg, "test")

    print("Extracting hidden video embeddings...")
    hidden_store, summary = perform_hidden_extraction(test_loader, model, cfg)

    save_path = Path(cfg.TEST.SAVE_PREDICT_PATH)
    save_path.parent.mkdir(parents=True, exist_ok=True)

    np.save(save_path, hidden_store)

    summary_path = save_path.with_suffix(".summary.json")
    summary_path.write_text(json.dumps(summary, indent=2))

    print("Saved hidden embeddings:", save_path)
    print("Saved summary:", summary_path)
    print(json.dumps(summary, indent=2))


def main():
    args = parse_args()
    cfg = load_config(args)
    launch_job(cfg=cfg, init_method=args.init_method, func=test)


if __name__ == "__main__":
    main()
'''

hidden_script_path.write_text(script)

print("Wrote:", hidden_script_path)
print("size:", hidden_script_path.stat().st_size)

# Sanity check: make sure the new script really contains the new logic.
text = hidden_script_path.read_text()
print("contains _choose_video_embedding_head:", "_choose_video_embedding_head" in text)
print("contains model.head preference:", '"model.head"' in text)
print("contains out_features == MODEL.NUM_CLASSES:", "out_features == MODEL.NUM_CLASSES" in text)
print("contains out_features == num_classes:", "out_features == num_classes" in text)

Wrote: /content/ProcedureVRL/tools/feat_extract_hidden.py
size: 8179
contains _choose_video_embedding_head: True
contains model.head preference: True
contains out_features == MODEL.NUM_CLASSES: False
contains out_features == num_classes: False


In [43]:
from pathlib import Path

hidden_script_path = Path("/content/ProcedureVRL/tools/feat_extract_hidden.py")

script = hidden_script_path.read_text()

# Patch hidden extractor:
# The current ProcedureVRL checkpoint does not expose a Linear head with out_features=9871.
# Instead, it has model.head = Linear(768 -> 512), which is the useful aligned video embedding projection.
old_choose = r'''
def _choose_hidden_head(model, num_classes):
    candidates = []

    for name, module in model.named_modules():
        if isinstance(module, nn.Linear):
            out_features = getattr(module, "out_features", None)
            in_features = getattr(module, "in_features", None)
            if out_features == num_classes:
                candidates.append({
                    "name": name,
                    "type": type(module).__name__,
                    "in_features": int(in_features) if in_features is not None else None,
                    "out_features": int(out_features),
                })

    # Prefer the final classification projection/head.
    def score(c):
        name = c["name"].lower()
        s = 0
        if "head" in name:
            s -= 20
        if "projection" in name or "proj" in name:
            s -= 10
        if "text" in name:
            s += 50
        return (s, len(name))

    candidates = sorted(candidates, key=score)

    if len(candidates) == 0:
        # Useful diagnostics if no exact final Linear is found.
        diagnostic = []
        for name, module in model.named_modules():
            if any(k in name.lower() for k in ["head", "projection", "proj", "classifier"]):
                diagnostic.append({
                    "name": name,
                    "type": type(module).__name__,
                    "repr": repr(module)[:300],
                })

        raise RuntimeError(
            "No nn.Linear module with out_features == MODEL.NUM_CLASSES was found. "
            "Diagnostics: " + json.dumps(diagnostic[:80], indent=2)
        )

    target = candidates[0]
    return target, candidates
'''

new_choose = r'''
def _choose_hidden_head(model, num_classes):
    candidates = []

    for name, module in model.named_modules():
        if isinstance(module, nn.Linear):
            out_features = getattr(module, "out_features", None)
            in_features = getattr(module, "in_features", None)

            candidates.append({
                "name": name,
                "type": type(module).__name__,
                "in_features": int(in_features) if in_features is not None else None,
                "out_features": int(out_features) if out_features is not None else None,
                "repr": repr(module)[:300],
            })

    # In this ProcedureVRL checkpoint, the useful aligned video projection is:
    #   model.head = Linear(768 -> 512)
    # It is not a 9871-way classification layer.
    for c in candidates:
        if c["name"] == "model.head" and c["out_features"] == 512:
            return c, candidates

    # Fallback: prefer any non-text head/projection producing 512-dim embeddings.
    filtered = [
        c for c in candidates
        if c["out_features"] == 512
        and "text_model" not in c["name"]
        and "order_tfm" not in c["name"]
    ]

    def score(c):
        name = c["name"].lower()
        s = 0
        if name == "model.head":
            s -= 100
        if "head" in name:
            s -= 20
        if "projection" in name or "proj" in name:
            s -= 10
        if "text" in name:
            s += 100
        return (s, len(name))

    filtered = sorted(filtered, key=score)

    if len(filtered) > 0:
        return filtered[0], candidates

    raise RuntimeError(
        "Could not find a suitable 512-dim video projection head. "
        "Linear candidates: " + json.dumps(candidates[:120], indent=2)
    )
'''

if old_choose not in script:
    print("Old _choose_hidden_head block not found exactly. Will do a simpler targeted replacement check.")
else:
    script = script.replace(old_choose, new_choose)

# Replace pre-hook with forward hook, because we want the OUTPUT of model.head:
#   768 -> model.head -> 512
old_hook = r'''
def _register_pre_head_hook(model, target_name):
    target_module = dict(model.named_modules())[target_name]

    def pre_hook(module, inputs):
        if len(inputs) == 0:
            return
        x = inputs[0]
        if isinstance(x, torch.Tensor):
            _CAPTURE["hidden"] = x.detach()
            _CAPTURE["raw_shape"] = list(x.shape)
            _CAPTURE["target_name"] = target_name

    handle = target_module.register_forward_pre_hook(pre_hook)
    return handle
'''

new_hook = r'''
def _register_pre_head_hook(model, target_name):
    target_module = dict(model.named_modules())[target_name]

    def forward_hook(module, inputs, output):
        x = output
        if isinstance(x, torch.Tensor):
            _CAPTURE["hidden"] = x.detach()
            _CAPTURE["raw_shape"] = list(x.shape)
            _CAPTURE["target_name"] = target_name
        elif isinstance(x, (tuple, list)) and len(x) > 0 and isinstance(x[0], torch.Tensor):
            _CAPTURE["hidden"] = x[0].detach()
            _CAPTURE["raw_shape"] = list(x[0].shape)
            _CAPTURE["target_name"] = target_name

    handle = target_module.register_forward_hook(forward_hook)
    return handle
'''

if old_hook not in script:
    print("Old hook block not found exactly.")
else:
    script = script.replace(old_hook, new_hook)

hidden_script_path.write_text(script)

print("Patched:", hidden_script_path)
print("Now the extractor should hook model.head output, i.e. 512-dim video embeddings.")

Old _choose_hidden_head block not found exactly. Will do a simpler targeted replacement check.
Old hook block not found exactly.
Patched: /content/ProcedureVRL/tools/feat_extract_hidden.py
Now the extractor should hook model.head output, i.e. 512-dim video embeddings.


## 7. Run hidden embedding extraction

In [44]:
%cd /content/ProcedureVRL

cmd = [
    sys.executable, "tools/feat_extract_hidden.py",
    "--cfg", "configs/HowTo100M/procedurevrl_adamw.yaml",

    "TRAIN.ENABLE", "False",
    "TRAIN.TEXT", "''",
    "DEV.ORDER_PRETRAIN_ENABLED", "False",

    "TEST.ENABLE", "True",
    "TEST.DATASET", "howto100m_develop",
    "TEST.CHECKPOINT_FILE_PATH", str(PROCEDUREVRL_CKPT_PATH),
    "TEST.SAVE_PREDICT_PATH", str(HIDDEN_NPY),
    "TEST.NUM_ENSEMBLE_VIEWS", str(cfg_run["num_ensemble_views"]),
    "TEST.NUM_SPATIAL_CROPS", "1",
    "TEST.BATCH_SIZE", str(cfg_run["batch_size"]),

    "DATA.PATH_TO_DATA_DIR", str(CSV_DIR),
    "DATA.PATH_PREFIX", str(RAW_VIDEO_BASE),
    "DATA.PATH_LABEL_SEPARATOR", " ",
    "DATA.NUM_FRAMES", "16",
    "DATA.SAMPLING_RATE", "6",
    "DATA.FD", "3.0",
    "DATA.TEST_CROP_SIZE", "224",
    "DATA.TRAIN_JITTER_SCALES", "[256,320]",
    "DATA.INPUT_CHANNEL_NUM", "[3]",
    "DATA.DECODING_BACKEND", "ffmpeg",

    "MODEL.NUM_CLASSES", "9871",
    "MODEL.HEAD_ACT", "softmax",

    "DATA_LOADER.NUM_WORKERS", str(cfg_run["num_workers"]),
    "DATA_LOADER.PIN_MEMORY", "True",

    "NUM_GPUS", "1",
    "NUM_SHARDS", "1",
    "OUTPUT_DIR", str(RUN_ROOT / "procedurevrl_hidden_output"),
    "LOG_MODEL_INFO", "False",
]

opts = cmd[4:]
print("num override args:", len(opts))
print("even:", len(opts) % 2 == 0)
assert len(opts) % 2 == 0

print("Command:")
print(" ".join(cmd))

start = time.time()

with open(RUN_LOG_PATH, "w") as log_f:
    process = subprocess.run(
        cmd,
        cwd=str(PROCEDUREVRL_REPO),
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        env={**os.environ, "PYTHONPATH": os.environ.get("PYTHONPATH", "")},
    )
    log_f.write(process.stdout)

elapsed = (time.time() - start) / 60.0

print("returncode:", process.returncode)
print("elapsed minutes:", elapsed)
print("log saved:", RUN_LOG_PATH)
print("\nLast 160 log lines:")
print("\n".join(process.stdout.splitlines()[-160:]))

if process.returncode != 0:
    raise RuntimeError("Hidden embedding extraction failed. Inspect the log above.")

/content/ProcedureVRL
num override args: 56
even: True
Command:
/usr/bin/python3 tools/feat_extract_hidden.py --cfg configs/HowTo100M/procedurevrl_adamw.yaml TRAIN.ENABLE False TRAIN.TEXT '' DEV.ORDER_PRETRAIN_ENABLED False TEST.ENABLE True TEST.DATASET howto100m_develop TEST.CHECKPOINT_FILE_PATH /content/drive/MyDrive/mmf_tas_lab_data/procedurevrl/checkpoints/checkpoint_epoch_00025.pyth TEST.SAVE_PREDICT_PATH /content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/breakfast/procedurevrl_hidden/runs/procedurevrl_hidden_full_split1_split1_views16/raw_outputs/procedurevrl_hidden_video_embeddings.npy TEST.NUM_ENSEMBLE_VIEWS 16 TEST.NUM_SPATIAL_CROPS 1 TEST.BATCH_SIZE 4 DATA.PATH_TO_DATA_DIR /content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/breakfast/procedurevrl_hidden/runs/procedurevrl_hidden_full_split1_split1_views16/data_csv DATA.PATH_PREFIX /content/drive/MyDrive/mmf_tas_lab_data/breakfast_raw_videos/videos DATA.PATH_LABEL_SEPARATOR   DATA.NUM_FRAMES 16 DATA.SAMPLING_RATE 6

## 8. Validate hidden embedding output

In [45]:
assert HIDDEN_NPY.exists(), HIDDEN_NPY

hidden = np.load(HIDDEN_NPY)
summary_path = HIDDEN_NPY.with_suffix(".summary.json")
summary = json.loads(summary_path.read_text()) if summary_path.exists() else {}

print("hidden shape:", hidden.shape)
print("dtype:", hidden.dtype)
print("min/max:", float(np.nanmin(hidden)), float(np.nanmax(hidden)))
print("nan count:", int(np.isnan(hidden).sum()))
print("summary:")
print(json.dumps(summary, indent=2))

expected_videos = len(df_run_manifest)
expected_views = cfg_run["num_ensemble_views"]

assert hidden.shape[0] == expected_videos, (hidden.shape[0], expected_videos)
assert hidden.shape[1] == expected_views, (hidden.shape[1], expected_views)
assert np.isfinite(hidden).all()

hidden_dim = int(hidden.shape[2])
print("hidden_dim:", hidden_dim)

hidden shape: (1712, 16, 512)
dtype: float32
min/max: -3.6244356632232666 6.030978679656982
nan count: 0
summary:
{
  "hook_target": {
    "name": "model.head",
    "type": "Linear",
    "in_features": 768,
    "out_features": 512,
    "repr": "Linear(in_features=768, out_features=512, bias=True)"
  },
  "n_video": 1712,
  "n_clip": 16,
  "hidden_dim": 512,
  "raw_hidden_shapes_seen": [
    [
      4,
      512
    ],
    [
      4,
      512
    ],
    [
      4,
      512
    ],
    [
      4,
      512
    ],
    [
      4,
      512
    ],
    [
      4,
      512
    ],
    [
      4,
      512
    ],
    [
      4,
      512
    ],
    [
      4,
      512
    ],
    [
      4,
      512
    ],
    [
      4,
      512
    ],
    [
      4,
      512
    ],
    [
      4,
      512
    ],
    [
      4,
      512
    ],
    [
      4,
      512
    ],
    [
      4,
      512
    ],
    [
      4,
      512
    ],
    [
      4,
      512
    ],
    [
      4,
      512
    ],
  

## 9. Convert hidden embeddings to MS-TCN-compatible dataset

In [46]:
def read_lines(path):
    return [x.strip() for x in Path(path).read_text().splitlines() if x.strip()]

def load_mapping(mapping_path):
    id_to_label = {}
    label_to_id = {}
    for line in read_lines(mapping_path):
        idx, label = line.split()[:2]
        id_to_label[int(idx)] = label
        label_to_id[label] = int(idx)
    return id_to_label, label_to_id

def downsample_labels_uniform(labels, target_len):
    if target_len <= 0:
        raise ValueError("target_len must be positive")
    if len(labels) == 0:
        return ["SIL"] * target_len
    positions = (np.arange(target_len) + 0.5) * len(labels) / target_len
    indices = np.clip(positions.astype(int), 0, len(labels) - 1)
    return [labels[int(i)] for i in indices]

id_to_label, label_to_id = load_mapping(ORIG_MAPPING_PATH)

shutil.copy2(ORIG_MAPPING_PATH, MSTCN_OUT_ROOT / "mapping.txt")

feature_rows = []
gt_rows = []

for i, row in df_run_manifest.iterrows():
    video_id = row["video_id"]
    split = row["split"]

    # hidden[i]: [T, D] -> MS-TCN convention [D, T]
    feat = hidden[i].astype(np.float32).T
    feat_path = MSTCN_FEATURE_DIR / f"{video_id}.npy"
    np.save(feat_path, feat)

    orig_labels = read_lines(ORIG_GT_DIR / f"{video_id}.txt")
    sampled_labels = downsample_labels_uniform(orig_labels, target_len=feat.shape[1])

    gt_path = MSTCN_GT_DIR / f"{video_id}.txt"
    gt_path.write_text("\n".join(sampled_labels) + "\n")

    feature_rows.append({
        "video_id": video_id,
        "split": split,
        "feature_path": str(feat_path),
        "feature_dim": int(feat.shape[0]),
        "feature_len": int(feat.shape[1]),
    })

    gt_rows.append({
        "video_id": video_id,
        "original_gt_len": len(orig_labels),
        "coarse_gt_len": len(sampled_labels),
        "feature_len": int(feat.shape[1]),
        "gt_path": str(gt_path),
    })

df_feature_manifest = pd.DataFrame(feature_rows)
df_gt_manifest = pd.DataFrame(gt_rows)

feature_manifest_path = RUN_ROOT / "hidden_mstcn_feature_manifest.csv"
gt_manifest_path = RUN_ROOT / "hidden_coarse_gt_manifest.csv"

df_feature_manifest.to_csv(feature_manifest_path, index=False)
df_gt_manifest.to_csv(gt_manifest_path, index=False)

for split_name in ["train", "test"]:
    ids = df_feature_manifest[df_feature_manifest["split"] == split_name]["video_id"].tolist()
    out_path = MSTCN_SPLIT_DIR / f"{split_name}.split{SPLIT_ID}.bundle"
    out_path.write_text("\n".join([f"{x}.txt" for x in ids]) + "\n")
    print(split_name, len(ids), "->", out_path)

display(df_feature_manifest.head())
display(df_gt_manifest.head())

print("features:", len(list(MSTCN_FEATURE_DIR.glob('*.npy'))))
print("groundTruth:", len(list(MSTCN_GT_DIR.glob('*.txt'))))
print("length mismatches:", int((df_gt_manifest["coarse_gt_len"] != df_gt_manifest["feature_len"]).sum()))
print("MSTCN_OUT_ROOT:", MSTCN_OUT_ROOT)

assert (df_gt_manifest["coarse_gt_len"] == df_gt_manifest["feature_len"]).all()

train 1460 -> /content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/breakfast/procedurevrl_hidden/runs/procedurevrl_hidden_full_split1_split1_views16/mstcn_format/splits/train.split1.bundle
test 252 -> /content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/breakfast/procedurevrl_hidden/runs/procedurevrl_hidden_full_split1_split1_views16/mstcn_format/splits/test.split1.bundle


,video_id,split,feature_path,feature_dim,feature_len
0,P16_cam01_P16_cereals,train,/content/drive/MyDrive/mmf_tas_lab_data/text_a...,512,16
1,P16_cam01_P16_friedegg,train,/content/drive/MyDrive/mmf_tas_lab_data/text_a...,512,16
2,P16_cam01_P16_juice,train,/content/drive/MyDrive/mmf_tas_lab_data/text_a...,512,16
3,P16_cam01_P16_milk,train,/content/drive/MyDrive/mmf_tas_lab_data/text_a...,512,16
4,P16_cam01_P16_pancake,train,/content/drive/MyDrive/mmf_tas_lab_data/text_a...,512,16


,video_id,original_gt_len,coarse_gt_len,feature_len,gt_path
0,P16_cam01_P16_cereals,544,16,16,/content/drive/MyDrive/mmf_tas_lab_data/text_a...
1,P16_cam01_P16_friedegg,6932,16,16,/content/drive/MyDrive/mmf_tas_lab_data/text_a...
2,P16_cam01_P16_juice,2238,16,16,/content/drive/MyDrive/mmf_tas_lab_data/text_a...
3,P16_cam01_P16_milk,884,16,16,/content/drive/MyDrive/mmf_tas_lab_data/text_a...
4,P16_cam01_P16_pancake,8040,16,16,/content/drive/MyDrive/mmf_tas_lab_data/text_a...


features: 1712
groundTruth: 1712
length mismatches: 0
MSTCN_OUT_ROOT: /content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/breakfast/procedurevrl_hidden/runs/procedurevrl_hidden_full_split1_split1_views16/mstcn_format
